# COMP8430 Phase 3 — Training & Evaluation Notebook

**Three deployment classes:** Cocacola_classic · Sprite · Fanta_orange

**Pipeline:**
1. Train on all 20 classes (Phase 2 clean dataset)
2. Evaluate on robot-captured test images — BEFORE fine-tuning
3. Fine-tune on robot-captured images (all 20 classes)
4. Evaluate again — AFTER fine-tuning
5. Final 3-class fine-tune for robot deployment
6. Before vs After comparison table and plots

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)

## 1. Dataset Overview — Phase 2 Clean Dataset

In [ ]:
clean_dataset = PROJECT_ROOT / '../dataset/clean_dataset'
class_counts = {}
for cls_dir in sorted(clean_dataset.iterdir()):
    if cls_dir.is_dir():
        n = sum(1 for f in cls_dir.iterdir() if f.suffix.lower() in {'.jpg','.jpeg','.png'})
        class_counts[cls_dir.name] = n

df = pd.DataFrame(list(class_counts.items()), columns=['Class','Images'])
df = df.sort_values('Images', ascending=False).reset_index(drop=True)
print(f'Total classes: {len(df)}  |  Total images: {df.Images.sum()}')
display(df)

plt.figure(figsize=(13,4))
plt.bar(df.Class, df.Images, color='steelblue')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.title('Images per class — clean_dataset (Phase 2)')
plt.ylabel('Count'); plt.tight_layout()
os.makedirs('../outputs/plots', exist_ok=True)
plt.savefig('../outputs/plots/dataset_distribution.png', dpi=150)
plt.show()

## 2. Three Selected Deployment Classes

**Selected:** Cocacola_classic, Sprite, Fanta_orange

**Reason:** These three classes are visually highly distinct from each other (red, green, orange colour schemes) which maximises classification accuracy and makes robot actions clearly distinguishable during the demo.

In [ ]:
THREE_CLASSES = ['Cocacola_classic', 'Sprite', 'Fanta_orange']
ACTIONS = {'Cocacola_classic':'Move forward','Sprite':'Rotate left','Fanta_orange':'Rotate right'}

print('Deployment classes and robot actions:')
for cls in THREE_CLASSES:
    n = sum(1 for f in (clean_dataset/cls).iterdir() if f.suffix.lower() in {'.jpg','.jpeg','.png'})
    print(f'  {cls}: {n} images → action: {ACTIONS[cls]}')

## 3. Robot-Captured Images Overview

Images captured during Week 10 practical using the robot's embedded camera.

In [ ]:
robot_captured = PROJECT_ROOT / '../dataset/robot_captured'
if robot_captured.exists():
    robot_counts = {}
    for cls_dir in sorted(robot_captured.iterdir()):
        if cls_dir.is_dir():
            n = sum(1 for f in cls_dir.iterdir() if f.suffix.lower() in {'.jpg','.jpeg','.png'})
            robot_counts[cls_dir.name] = n
    df_robot = pd.DataFrame(list(robot_counts.items()), columns=['Class','Robot Images'])
    df_robot = df_robot.sort_values('Robot Images', ascending=False).reset_index(drop=True)
    print(f'Robot classes: {len(df_robot)}  |  Total images: {df_robot["Robot Images"].sum()}')
    display(df_robot)
else:
    print('Robot captured images not found — place in ../dataset/robot_captured/')

## 4. Training Curves — All 20 Classes (Phase 2 Dataset)

In [ ]:
path = '../outputs/plots/all_class_train_curves.png'
if os.path.exists(path):
    plt.figure(figsize=(11,4)); plt.imshow(mpimg.imread(path)); plt.axis('off')
    plt.title('All-class training — Phase 2 clean dataset'); plt.show()
else:
    print('Run: python train_all_classes.py --data-dir ../dataset/clean_dataset --epochs 20')

## 5. Evaluation BEFORE Robot Fine-Tuning

Test the Phase 2 model on robot-captured test images.

In [ ]:
path = '../outputs/logs/before_robot_finetune_metrics.json'
if os.path.exists(path):
    m = json.load(open(path))
    print('=== Before robot fine-tune ===')
    print(f'  Accuracy  : {m["accuracy"]:.4f}')
    print(f'  F1 Macro  : {m["f1_macro"]:.4f}')
    print(f'  F1 Weighted: {m["f1_weighted"]:.4f}')
    print('\n  Per-class accuracy (3 deployment classes):')
    for cls in THREE_CLASSES:
        acc = m['per_class_accuracy'].get(cls, 'N/A')
        print(f'    {cls}: {acc}')
else:
    print('Run: python evaluate.py --data-dir ../dataset/robot_split/test --model-path outputs/models/all_class_model.pth --tag before_robot_finetune')

## 6. Training Curves — Robot Fine-Tuning (All 20 Classes)

In [ ]:
path = '../outputs/plots/robot_finetune_curves.png'
if os.path.exists(path):
    plt.figure(figsize=(11,4)); plt.imshow(mpimg.imread(path)); plt.axis('off')
    plt.title('Robot fine-tuning curves — all 20 classes'); plt.show()
else:
    print('Run: python train_robot_finetune.py')

## 7. Evaluation AFTER Robot Fine-Tuning

In [ ]:
path = '../outputs/logs/after_robot_finetune_metrics.json'
if os.path.exists(path):
    m = json.load(open(path))
    print('=== After robot fine-tune ===')
    print(f'  Accuracy  : {m["accuracy"]:.4f}')
    print(f'  F1 Macro  : {m["f1_macro"]:.4f}')
    print(f'  F1 Weighted: {m["f1_weighted"]:.4f}')
    print('\n  Per-class accuracy (3 deployment classes):')
    for cls in THREE_CLASSES:
        acc = m['per_class_accuracy'].get(cls, 'N/A')
        print(f'    {cls}: {acc}')
else:
    print('Run: python evaluate.py --data-dir ../dataset/robot_split/test --model-path outputs/models/robot_finetuned_model.pth --tag after_robot_finetune')

## 8. Before vs After Comparison Table

In [ ]:
b_path = '../outputs/logs/before_robot_finetune_metrics.json'
a_path = '../outputs/logs/after_robot_finetune_metrics.json'
if os.path.exists(b_path) and os.path.exists(a_path):
    b = json.load(open(b_path)); a = json.load(open(a_path))
    comp = {
        'Model': ['Before robot fine-tune','After robot fine-tune'],
        'Accuracy':    [b['accuracy'],    a['accuracy']],
        'F1 Macro':    [b['f1_macro'],    a['f1_macro']],
        'F1 Weighted': [b['f1_weighted'], a['f1_weighted']],
        'Precision':   [b['precision_macro'], a['precision_macro']],
        'Recall':      [b['recall_macro'],    a['recall_macro']],
    }
    df_comp = pd.DataFrame(comp)
    display(df_comp.style.highlight_max(subset=['Accuracy','F1 Macro','F1 Weighted'], color='lightgreen'))
    
    # 3 deployment classes per-class comparison
    print('\nPer-class accuracy for 3 deployment classes:')
    rows = []
    for cls in THREE_CLASSES:
        rows.append({'Class': cls,
                     'Before': round(b['per_class_accuracy'].get(cls,0),4),
                     'After':  round(a['per_class_accuracy'].get(cls,0),4)})
    df3 = pd.DataFrame(rows)
    df3['Change'] = (df3['After'] - df3['Before']).round(4)
    display(df3)
else:
    print('Run scripts 3 and 5 first.')

In [ ]:
# Comparison plot
path = '../outputs/plots/before_after_per_class.png'
if os.path.exists(path):
    plt.figure(figsize=(12,5)); plt.imshow(mpimg.imread(path)); plt.axis('off')
    plt.title('Per-class accuracy before vs after robot fine-tuning'); plt.show()

## 9. Confusion Matrices

In [ ]:
for tag, title in [('before_robot_finetune','Before fine-tune'),('after_robot_finetune','After fine-tune')]:
    path = f'../outputs/plots/{tag}_confusion_matrix.png'
    if os.path.exists(path):
        plt.figure(figsize=(12,5)); plt.imshow(mpimg.imread(path)); plt.axis('off')
        plt.title(f'Confusion Matrix — {title}'); plt.show()

## 10. Three-Class Robot Model — Final Results

Fine-tuned specifically on Cocacola_classic, Sprite, Fanta_orange.

In [ ]:
path = '../outputs/plots/three_class_train_curves.png'
if os.path.exists(path):
    plt.figure(figsize=(11,4)); plt.imshow(mpimg.imread(path)); plt.axis('off')
    plt.title('Three-class fine-tuning curves'); plt.show()
else:
    print('Run: python train_three_classes.py')

## 11. Robot Actions Summary

| Class | Action | Twist |
|-------|--------|-------|
| Cocacola_classic | Move forward toward object | linear.x=0.1, angular.z=small correction |
| Sprite | Rotate left (circle around object) | linear.x=0.0, angular.z=+0.5 |
| Fanta_orange | Rotate right (circle around object) | linear.x=0.0, angular.z=-0.5 |

**State machine:** SEARCH (rotate 0.3) → ALIGN (centre object) → ACTION (class-specific) → ARRIVED/DONE

## 12. Discussion

### Why these three classes?
Cocacola_classic, Sprite, and Fanta_orange were selected because they are visually highly distinct (red, green, and orange colour schemes respectively). This maximises classification confidence and makes the three robot actions clearly distinguishable during the demo video.

### Why split robot images into fine-tune and test sets?
Robot-captured images differ from the studio Phase 2 dataset due to varying lighting, motion blur, camera angle, and background. Splitting into fine-tune and test ensures the model learns robot-specific characteristics while the test set provides an unbiased measure of real-world generalisation.

### Why transfer learning?
MobileNetV3-Large pretrained on ImageNet provides strong low-level features (edges, textures, shapes) that apply to any image domain. Only the classifier head needs retraining, making fine-tuning fast and effective even with a small robot-captured image count.

### ROI detection limitation
Bounding box detection uses OpenCV contour extraction (Canny edges + dilation + findContours). This works without bounding-box annotations but is sensitive to background clutter. In controlled lab conditions it performs reliably. A trained detector (e.g. YOLOv8-nano) would improve robustness.

### Future improvements
- Train a YOLOv8-nano object detector with annotated bounding boxes
- Use depth camera data to precisely control approach distance
- Expand deployment classes to all 20 for a more complete demo